In [1]:
import tensorflow as tf
import numpy as np

2026-07-26 14:54:27.813185: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.



# 1. SYNTHETIC LOAN DATA
In practice, you would load a real dataset here:
  - data = np.genfromtxt("loan_data.csv", delimiter=",", skip_header=1)
  - X = data[:, :-1]  
  - y = data[:, -1]    

Each row represents one loan application.
Features: monthly income, credit score, debt-to-income ratio, years employed
Label:    1 = approved, 0 = denied


In [2]:
np.random.seed(42)
n = 500

# Approved applicants: higher income, higher credit, lower debt, more experience
approved_income   = np.random.normal(loc=6000, scale=1500, size=n // 2)
approved_credit   = np.random.normal(loc=720, scale=40, size=n // 2)
approved_debt     = np.random.normal(loc=0.25, scale=0.08, size=n // 2)
approved_years    = np.random.normal(loc=6, scale=2, size=n // 2)

# Denied applicants: lower income, lower credit, higher debt, less experience
denied_income   = np.random.normal(loc=2500, scale=1000, size=n // 2)
denied_credit   = np.random.normal(loc=580, scale=50, size=n // 2)
denied_debt     = np.random.normal(loc=0.55, scale=0.1, size=n // 2)
denied_years    = np.random.normal(loc=2, scale=1.5, size=n // 2)

X = np.column_stack([
    np.concatenate([approved_income, denied_income]),
    np.concatenate([approved_credit, denied_credit]),
    np.concatenate([approved_debt, denied_debt]),
    np.concatenate([approved_years, denied_years])
]).astype(np.float32)

y = np.concatenate([
    np.ones(n // 2),
    np.zeros(n // 2)
]).astype(np.float32).reshape(-1, 1)

# Shuffle so approved and denied cases are mixed
shuffle = np.random.permutation(n)
X, y = X[shuffle], y[shuffle]

# 2. NORMALIZE FEATURES

Neural networks train faster and more reliably when inputs
are on a similar scale. Income (2000-8000) and debt ratio
(0.1-0.8) would otherwise fight for influence.

In [3]:
mean = np.mean(X, axis=0)
std  = np.std(X, axis=0)
X = (X - mean) / std

# Split into training and test sets
split = int(0.8 * n)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Training samples: {len(y_train)}")
print(f"Test samples:     {len(y_test)}\n")

Training samples: 400
Test samples:     100



# 3. BUILD THE MODEL

Same structure as the AND gate code: input → hidden → output.
The difference is that the input now has 4 features instead of 2,
and the hidden layer has 8 neurons instead of 4 to handle the
added complexity.

In [20]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(4,)),
    tf.keras.layers.Dense(8, activation='tanh'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 4. PREDICTIONS BEFORE TRAINING


In [21]:
print("Predictions BEFORE training:")
print(f"  {'Income':>8} {'Credit':>8} {'Debt':>6} {'Years':>6} {'Output':>8} {'Decision':>10}")
print(f"  {'─'*8} {'─'*8} {'─'*6} {'─'*6} {'─'*8} {'─'*10}")

sample_raw = X_test[:6]
labels_raw = y_test[:6].flatten()
pre_pred   = model.predict(sample_raw, verbose=0).flatten()

for features, pred, actual in zip(sample_raw, pre_pred, labels_raw):
    # Denormalize to show real values
    real = features * std + mean
    decision = "Approved" if pred >= 0.5 else "Denied"
    print(f"  {real[0]:>8.0f} {real[1]:>8.0f} {real[2]:>6.2f} {real[3]:>5.1f}  {pred:>7.4f}  {decision:>10} (actual: {'Approved' if actual else 'Denied'})")

Predictions BEFORE training:
    Income   Credit   Debt  Years   Output   Decision
  ──────── ──────── ────── ────── ──────── ──────────
      6138      745   0.13   5.2   0.1652      Denied (actual: Approved)
      6515      753   0.28   6.1   0.2555      Denied (actual: Approved)
      7581      740   0.14   8.9   0.4237      Denied (actual: Approved)
      2309      579   0.52   1.8   0.6496    Approved (actual: Denied)
      2534      683   0.65   2.0   0.4598      Denied (actual: Denied)
      4514      755   0.17   5.0   0.1922      Denied (actual: Approved)


# 5. TRAIN THE MODEL


In [22]:
print("\nTraining...")
history = model.fit(X_train, y_train, epochs=50, verbose=0)
print(f"  Final training accuracy: {history.history['accuracy'][-1]:.1%}\n")



Training...
  Final training accuracy: 99.8%



# 6. EVALUATE ON TEST SET


In [23]:

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
y_pred_prob = model.predict(X_test, verbose=0).flatten()
y_pred = (y_pred_prob >= 0.5).astype(int)
y_true = y_test.flatten().astype(int)

tp = np.sum((y_pred == 1) & (y_true == 1))
tn = np.sum((y_pred == 0) & (y_true == 0))
fp = np.sum((y_pred == 1) & (y_true == 0))
fn = np.sum((y_pred == 0) & (y_true == 1))

print(f"  Test Accuracy: {test_acc:.1%}")
print(f"  Precision:     {tp / (tp + fp):.1%}")
print(f"  Recall:        {tp / (tp + fn):.1%}")

print(f"\n  Confusion Matrix:")
print(f"                   Predicted")
print(f"                   Denied  Approved")
print(f"  Actual Denied  [  {tn:>4}     {fp:>4}  ]")
print(f"  Actual Approved[  {fn:>4}     {tp:>4}  ]")

  Test Accuracy: 99.0%
  Precision:     98.1%
  Recall:        100.0%

  Confusion Matrix:
                   Predicted
                   Denied  Approved
  Actual Denied  [    46        1  ]
  Actual Approved[     0       53  ]


# 7. PREDICT NEW APPLICANTS


In [19]:
new_applicants = np.array([
    [7000, 740, 0.20, 8],    # strong applicant
    [2000, 550, 0.70, 1],    # risky applicant
    [4500, 650, 0.35, 4],    # borderline applicant
    [5500, 690, 0.30, 5],    # decent applicant
    [1800, 500, 0.80, 0.5],  # very risky
], dtype=np.float32)

# Normalize using the same statistics from training
new_normalized = (new_applicants - mean) / std
new_predictions = model.predict(new_normalized, verbose=0).flatten()

print(f"\n  New Loan Applications:")
print(f"  {'Income':>8} {'Credit':>8} {'Debt':>6} {'Years':>6} {'Confidence':>12} {'Decision':>10}")
print(f"  {'─'*8} {'─'*8} {'─'*6} {'─'*6} {'─'*12} {'─'*10}")

for applicant, pred in zip(new_applicants, new_predictions):
    decision = "Approved" if pred >= 0.5 else "Denied"
    confidence = pred if pred >= 0.5 else 1 - pred
    print(f"  {applicant[0]:>8.0f} {applicant[1]:>8.0f} {applicant[2]:>6.2f} {applicant[3]:>5.1f} {confidence:>11.1%} {decision:>10}")


  New Loan Applications:
    Income   Credit   Debt  Years   Confidence   Decision
  ──────── ──────── ────── ────── ──────────── ──────────
      7000      740   0.20   8.0      100.0%   Approved
      2000      550   0.70   1.0      100.0%     Denied
      4500      650   0.35   4.0       76.3%   Approved
      5500      690   0.30   5.0       99.4%   Approved
      1800      500   0.80   0.5      100.0%     Denied
